In [2]:
#r "nuget: ScottPlot, 5.0.39"

Installed Packages ScottPlot, 5.0.39

Loading extensions from `C:\Users\user\.nuget\packages\skiasharp\2.88.8\interactive-extensions\dotnet\SkiaSharp.DotNet.Interactive.dll`

In [ ]:
#r "bin/Debug/net10.0/task14.dll"
using System;
using System.Diagnostics;
using System.Collections.Generic;
using System.Linq;
using ScottPlot;
using System.IO;
using task14;

double a = -100;
double b = 100;
Func<double, double> func = Math.Sin;

double[] steps = { 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6 };
int measurements = 5;
double requiredPrecision = 1e-4;
double speedupThreshold = 15.0;

double intervalLength = b - a;
double maxSecondDerivative = 1.0;

double chosenStep = 0;
int optimalThreads = 0;
double singleTime = 0;
double bestMultiTime = 0;
double finalSpeedup = 0;
List<double> chosenTimes = null;
List<int> chosenThreadCounts = null;

foreach (double step in steps)
{
    double errorBound = intervalLength / 12.0 * step * step * maxSecondDerivative;
    Console.WriteLine($"=== Шаг {step:E1} ===");
    Console.WriteLine($"Теоретическая оценка погрешности: {errorBound:E2} (требуется <= {requiredPrecision:E1})");

    if (errorBound > requiredPrecision)
    {
        Console.WriteLine("-> точность не обеспечивается, пропускаем шаг.\n");
        continue;
    }

    double singleStepTime = 0;
    for (int i = 0; i < measurements; i++)
    {
        Stopwatch sw = Stopwatch.StartNew();
        DefiniteIntegral.SolveSingleThread(a, b, func, step);
        sw.Stop();
        singleStepTime += sw.Elapsed.TotalMilliseconds;
    }
    singleStepTime /= measurements;
    Console.WriteLine($"Однопоточное время: {singleStepTime:F2} мс");

    List<double> times = new();
    List<int> threadCounts = new();
    double bestTime = double.MaxValue;
    int bestThreadsForStep = 0;

    for (int threads = 1; threads <= 16; threads++)
    {
        double avg = 0;
        for (int i = 0; i < measurements; i++)
        {
            Stopwatch sw = Stopwatch.StartNew();
            DefiniteIntegral.Solve(a, b, func, step, threads);
            sw.Stop();
            avg += sw.Elapsed.TotalMilliseconds;
        }
        avg /= measurements;
        times.Add(avg);
        threadCounts.Add(threads);
        Console.WriteLine($"  Потоков {threads}: {avg:F2} мс");

        if (avg < bestTime)
        {
            bestTime = avg;
            bestThreadsForStep = threads;
        }
    }

    double speedup = ((singleStepTime - bestTime) / singleStepTime) * 100;
    Console.WriteLine($"Лучшее многопоточное время: {bestTime:F2} мс при {bestThreadsForStep} потоках");
    Console.WriteLine($"Ускорение: {speedup:F2}%");

    if (speedup > speedupThreshold)
    {
        chosenStep = step;
        optimalThreads = bestThreadsForStep;
        singleTime = singleStepTime;
        bestMultiTime = bestTime;
        finalSpeedup = speedup;
        chosenTimes = times;
        chosenThreadCounts = threadCounts;
        Console.WriteLine($"✓ Шаг {step:E1} подходит: точность есть, ускорение {speedup:F2}% > {speedupThreshold}%. Останавливаемся.\n");
        break;
    }
    else
    {
        Console.WriteLine($"✗ Ускорение {speedup:F2}% <= {speedupThreshold}%.");
        Console.WriteLine($"Следующий шаг ({step / 10:E1}) даст в ~10 раз больше итераций, поэтому даже однопоточное время вырастет пропорционально - переходим к нему, только если это единственный способ добиться нужного ускорения.\n");
    }
}

if (chosenStep == 0)
{
    Console.WriteLine("Не найден шаг, который одновременно удовлетворяет точности и даёт ускорение > 15%.");
    return;
}

Console.WriteLine("=== ИТОГ ===");
Console.WriteLine($"Выбранный шаг: {chosenStep:E1}");
Console.WriteLine($"Оптимальное число потоков: {optimalThreads}");
Console.WriteLine($"Однопоточное время: {singleTime:F2} мс");
Console.WriteLine($"Многопоточное время: {bestMultiTime:F2} мс");
Console.WriteLine($"Ускорение: {finalSpeedup:F2}%");

var plt = new ScottPlot.Plot();
double[] plotX = chosenTimes.ToArray();
double[] plotY = chosenThreadCounts.Select(t => (double)t).ToArray();
var scatter = plt.Add.Scatter(plotX, plotY);
scatter.LineWidth = 3;
scatter.MarkerSize = 10;
plt.Title($"Время выполнения Solve от количества потоков (шаг {chosenStep:E1})");
plt.XLabel("Время выполнения (мс)");
plt.YLabel("Количество потоков");
plt.SavePng("result.png", 800, 600);

File.WriteAllText("result.txt",
$"""

Выбор шага и потоков:
Шаг выбирался перебором от грубого к точному: для каждого шага сначала проверялась
теоретическая оценка погрешности метода трапеций |E| <= (b-a)/12 * h^2 * max|f''(x)|
(прямое сравнение с точным интегралом на [-100,100] не показательно из-за симметрии
задачи - подробности в начале скрипта). Если шаг проходил по точности, для него
измерялось однопоточное время и время при разном числе потоков (1..16), считалось
ускорение. Выбирался первый (самый грубый, а значит самый быстрый по вычислениям) шаг,
для которого ускорение многопоточной версии превысило {speedupThreshold}%.
Более мелкие шаги дают в разы больше итераций (10x на каждый порядок), поэтому даже
при лучшем ускорении их абсолютное время работы оказывается заметно выше.

Выбранный шаг: {chosenStep:E1}
Оптимальное число потоков: {optimalThreads}

Производительность:
Однопоточное время: {singleTime:F2} мс
Многопоточное время: {bestMultiTime:F2} мс
Ускорение: {finalSpeedup:F2}%
""");

Console.WriteLine("Готово");

=== Шаг 1.0E-001 ===
Теоретическая оценка погрешности: 1.67E-001 (требуется <= 1.0E-004)
-> точность не обеспечивается, пропускаем шаг.

=== Шаг 1.0E-002 ===
Теоретическая оценка погрешности: 1.67E-003 (требуется <= 1.0E-004)
-> точность не обеспечивается, пропускаем шаг.

=== Шаг 1.0E-003 ===
Теоретическая оценка погрешности: 1.67E-005 (требуется <= 1.0E-004)
Однопоточное время: 2.78 мс
  Потоков 1: 3.69 мс
  Потоков 2: 2.22 мс
  Потоков 3: 1.81 мс
  Потоков 4: 1.57 мс
  Потоков 5: 1.60 мс
  Потоков 6: 1.63 мс
  Потоков 7: 3.13 мс
  Потоков 8: 1.80 мс
  Потоков 9: 1.78 мс
  Потоков 10: 2.09 мс
  Потоков 11: 2.18 мс
  Потоков 12: 2.32 мс
  Потоков 13: 2.52 мс
  Потоков 14: 2.87 мс
  Потоков 15: 3.22 мс
  Потоков 16: 3.13 мс
Лучшее многопоточное время: 1.57 мс при 4 потоках
Ускорение: 43.50%
✓ Шаг 1.0E-003 подходит: точность есть, ускорение 43.50% > 15%. Останавливаемся.

=== ИТОГ ===
Выбранный шаг: 1.0E-003
Оптимальное число потоков: 4
Однопоточное время: 2.78 мс
Многопоточное время: 1

<null>